# Clinical Triage Model Demo

This notebook will be used to test locally installed open-source models for clinical triage estimation.

For now, it focuses only on dataset cleaning:

- keep only triage-available text, demographic, vital-sign, and pain-score fields
- extract `esi_level` as the prediction label
- save feature and label files for later model testing

No model evaluation is performed in this first version.

In [1]:
from pathlib import Path
import os

import pandas as pd

def find_project_root(start: Path, marker: str = "fedmml_ed_triage_dataset.csv") -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {marker}. Start Jupyter from the repository root "
        "or place the CSV beside this notebook."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

os.chdir(PROJECT_ROOT)
RAW_CSV = PROJECT_ROOT / "fedmml_ed_triage_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV


WindowsPath('c:/Users/money/OneDrive/Documents/GitHub/oss_model_clinical_triage_demo/fedmml_ed_triage_dataset.csv')

## 1. Load Raw Data

In [2]:
raw_df = pd.read_csv(RAW_CSV)
raw_df.shape


(87234, 28)

In [3]:
raw_df.head()


,encounter_id,patient_id,site_id,country,age,sex,arrival_timestamp,chief_complaint,clinical_notes,systolic_bp,...,platelet_count,sodium,potassium,creatinine,glucose,troponin,bnp,lactate,inr,esi_level
0,ENC1000001,PAT000001,1,Denmark,59,F,2021-01-30 12:33:00,Back pain,NaN,122.0,...,244.0,142.1,3.77,0.74,75.0,0.0,65.0,1.13,1.01,3
1,ENC1000002,PAT000002,1,Denmark,67,M,2022-02-26 01:46:00,Medication question,67yo M requesting Medication question. Patient...,138.0,...,242.0,138.4,4.09,1.15,84.0,0.0,85.0,1.24,0.68,5
2,ENC1000003,PAT000003,1,Denmark,58,F,2021-10-20 08:55:00,Cold symptoms,"58yo F here for Cold symptoms. Patient stable,...",114.0,...,296.0,139.0,3.82,1.11,111.0,0.0,56.0,0.94,1.12,4
3,ENC1000004,PAT000004,1,Denmark,23,F,2021-01-11 20:11:00,Laceration requiring sutures,23yo F presents with Laceration requiring sutu...,137.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
4,ENC1000005,PAT000005,1,Denmark,64,F,2023-11-05 13:44:00,Chest pain,64yo F c/o Chest pain. Patient in moderate dis...,141.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2


In [4]:
raw_df.columns.tolist()


['encounter_id',
 'patient_id',
 'site_id',
 'country',
 'age',
 'sex',
 'arrival_timestamp',
 'chief_complaint',
 'clinical_notes',
 'systolic_bp',
 'diastolic_bp',
 'heart_rate',
 'respiratory_rate',
 'temperature',
 'spo2',
 'pain_score',
 'wbc',
 'hemoglobin',
 'platelet_count',
 'sodium',
 'potassium',
 'creatinine',
 'glucose',
 'troponin',
 'bnp',
 'lactate',
 'inr',
 'esi_level']

## 2. Validate Required Columns

In [5]:
FEATURE_COLUMNS = [
    "age",
    "sex",
    "chief_complaint",
    "clinical_notes",
    "systolic_bp",
    "diastolic_bp",
    "heart_rate",
    "respiratory_rate",
    "temperature",
    "spo2",
    "pain_score",
]
LABEL_COLUMN = "esi_level"

required_columns = FEATURE_COLUMNS + [LABEL_COLUMN]
missing_required = [column for column in required_columns if column not in raw_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

raw_df[required_columns].head()


,age,sex,chief_complaint,clinical_notes,systolic_bp,diastolic_bp,heart_rate,respiratory_rate,temperature,spo2,pain_score,esi_level
0,59,F,Back pain,NaN,122.0,88.0,80.0,16.0,38.2,97.2,6.0,3
1,67,M,Medication question,67yo M requesting Medication question. Patient...,138.0,68.0,82.0,18.0,36.7,98.4,2.0,5
2,58,F,Cold symptoms,"58yo F here for Cold symptoms. Patient stable,...",114.0,76.0,85.0,16.0,37.6,97.8,1.0,4
3,23,F,Laceration requiring sutures,23yo F presents with Laceration requiring sutu...,137.0,81.0,79.0,20.0,38.0,91.6,4.0,3
4,64,F,Chest pain,64yo F c/o Chest pain. Patient in moderate dis...,141.0,71.0,99.0,21.0,37.8,98.8,8.0,2


## 3. Select Triage Columns And Extract Label

`esi_level` is the target label. The feature table keeps only the accepted triage columns and excludes identifiers, site/country fields, timestamps, and lab values.

In [6]:
modeling_df = raw_df[FEATURE_COLUMNS + [LABEL_COLUMN]].dropna(axis=0, how="any").copy()
y = modeling_df[LABEL_COLUMN].astype("int64").rename("label_esi_level")
X = modeling_df[FEATURE_COLUMNS].copy()

print("Raw shape:", raw_df.shape)
print("Rows after dropping missing values:", len(modeling_df))
print("Rows removed:", len(raw_df) - len(modeling_df))
print("Feature shape:", X.shape)
print("Label shape:", y.shape)
print("Kept feature columns:", FEATURE_COLUMNS)
print("Label column:", LABEL_COLUMN)


Raw shape: (87234, 28)
Rows after dropping missing values: 80275
Rows removed: 6959
Feature shape: (80275, 11)
Label shape: (80275,)
Kept feature columns: ['age', 'sex', 'chief_complaint', 'clinical_notes', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature', 'spo2', 'pain_score']
Label column: esi_level


In [7]:
assert X.columns.tolist() == FEATURE_COLUMNS
assert LABEL_COLUMN not in X.columns
assert len(X) == len(y)

X.head()


,age,sex,chief_complaint,clinical_notes,systolic_bp,diastolic_bp,heart_rate,respiratory_rate,temperature,spo2,pain_score
1,67,M,Medication question,67yo M requesting Medication question. Patient...,138.0,68.0,82.0,18.0,36.7,98.4,2.0
2,58,F,Cold symptoms,"58yo F here for Cold symptoms. Patient stable,...",114.0,76.0,85.0,16.0,37.6,97.8,1.0
3,23,F,Laceration requiring sutures,23yo F presents with Laceration requiring sutu...,137.0,81.0,79.0,20.0,38.0,91.6,4.0
4,64,F,Chest pain,64yo F c/o Chest pain. Patient in moderate dis...,141.0,71.0,99.0,21.0,37.8,98.8,8.0
5,46,F,High fever with confusion,46yo F c/o High fever with confusion. Patient ...,162.0,102.0,133.0,26.0,38.1,89.7,8.0


## 4. Label Distribution

Emergency Severity Index (ESI) usually ranges from 1 to 5, where lower numbers indicate higher acuity.

In [8]:
label_distribution = y.value_counts().sort_index().rename_axis("esi_level").reset_index(name="count")
label_distribution["fraction"] = label_distribution["count"] / len(y)
label_distribution


,esi_level,count,fraction
0,1,862,0.010738
1,2,15496,0.193036
2,3,38062,0.474145
3,4,21319,0.265575
4,5,4536,0.056506


## 5. Missingness Check

Rows with any missing values were excluded for now. This cell verifies that the saved modeling features have no missing values remaining.

In [9]:
missingness = X.isna().sum().sort_values(ascending=False).rename("missing_count").reset_index()
missingness.columns = ["column", "missing_count"]
missingness["missing_fraction"] = missingness["missing_count"] / len(X)
missingness.head(20)


,column,missing_count,missing_fraction
0,age,0,0.0
1,sex,0,0.0
2,chief_complaint,0,0.0
3,clinical_notes,0,0.0
4,systolic_bp,0,0.0
5,diastolic_bp,0,0.0
6,heart_rate,0,0.0
7,respiratory_rate,0,0.0
8,temperature,0,0.0
9,spo2,0,0.0


## 6. Save Cleaned Outputs

The cleaned files are saved under `data/processed/`:

- `clinical_triage_features.csv`: selected triage features only, no label
- `clinical_triage_labels.csv`: ESI label only
- `clinical_triage_cleaned_with_label.csv`: convenience file with features plus label

In [10]:
features_path = PROCESSED_DIR / "clinical_triage_features.csv"
labels_path = PROCESSED_DIR / "clinical_triage_labels.csv"
combined_path = PROCESSED_DIR / "clinical_triage_cleaned_with_label.csv"

X.to_csv(features_path, index=False)
y.to_frame().to_csv(labels_path, index=False)
pd.concat([X, y], axis=1).to_csv(combined_path, index=False)

print(features_path)
print(labels_path)
print(combined_path)


c:\Users\money\OneDrive\Documents\GitHub\oss_model_clinical_triage_demo\data\processed\clinical_triage_features.csv
c:\Users\money\OneDrive\Documents\GitHub\oss_model_clinical_triage_demo\data\processed\clinical_triage_labels.csv
c:\Users\money\OneDrive\Documents\GitHub\oss_model_clinical_triage_demo\data\processed\clinical_triage_cleaned_with_label.csv


In [11]:
check_features = pd.read_csv(features_path, nrows=5)
check_labels = pd.read_csv(labels_path, nrows=5)
check_combined = pd.read_csv(combined_path, nrows=5)

print("features columns match selected set:", check_features.columns.tolist() == FEATURE_COLUMNS)
print("features contains esi_level:", LABEL_COLUMN in check_features.columns)
print("labels columns:", check_labels.columns.tolist())
print("combined columns tail:", check_combined.columns[-5:].tolist())

check_features.head()


features columns match selected set: True
features contains esi_level: False
labels columns: ['label_esi_level']
combined columns tail: ['respiratory_rate', 'temperature', 'spo2', 'pain_score', 'label_esi_level']


,age,sex,chief_complaint,clinical_notes,systolic_bp,diastolic_bp,heart_rate,respiratory_rate,temperature,spo2,pain_score
0,67,M,Medication question,67yo M requesting Medication question. Patient...,138.0,68.0,82.0,18.0,36.7,98.4,2.0
1,58,F,Cold symptoms,"58yo F here for Cold symptoms. Patient stable,...",114.0,76.0,85.0,16.0,37.6,97.8,1.0
2,23,F,Laceration requiring sutures,23yo F presents with Laceration requiring sutu...,137.0,81.0,79.0,20.0,38.0,91.6,4.0
3,64,F,Chest pain,64yo F c/o Chest pain. Patient in moderate dis...,141.0,71.0,99.0,21.0,37.8,98.8,8.0
4,46,F,High fever with confusion,46yo F c/o High fever with confusion. Patient ...,162.0,102.0,133.0,26.0,38.1,89.7,8.0


The models receive a patient case as JSON text. The MedGemma cells score the allowed values for `predicted_esi_level`: `1`, `2`, `3`, `4`, `5`, or `"uncertain"`.


In [12]:
from clinical_triage_utils import (
    PROMPT_FEATURE_COLUMNS,
    build_esi_prompt,
    case_json_to_retrieval_text,
    embed_texts_medsiglip,
    empty_result_counter,
    is_valid_model_input,
    generate_medgemma_4b_prediction,
    generate_text_model_prediction,
    row_to_case_json,
    score_prediction,
)

PROMPT_TEMPLATE_PATH = PROJECT_ROOT / "prompts" / "esi_prediction_prompt.txt"
PROMPT_TEMPLATE_PATH.exists(), PROMPT_TEMPLATE_PATH

(True,
 WindowsPath('c:/Users/money/OneDrive/Documents/GitHub/oss_model_clinical_triage_demo/prompts/esi_prediction_prompt.txt'))

The MedGemma cells use constrained-choice scoring: the model sees the patient JSON and scores only the allowed values for `predicted_esi_level`.

The notebook then formats the selected value as:

```json
{"predicted_esi_level": 1}
```

or, when the model selects uncertainty:

```json
{"predicted_esi_level": "uncertain"}
```

## 8. Runtime Settings

Choose CPU/GPU behavior before running the model cells. By default this notebook uses CPU, which is safest for broad compatibility. If you install a CUDA-enabled PyTorch build, set `FORCE_CPU = False` to use GPU automatically.

In [ ]:
import sys
import torch

# Keep True for maximum compatibility. Set False only if CUDA PyTorch is installed and working.
FORCE_CPU = False

GPU_AVAILABLE = torch.cuda.is_available()
USE_GPU = GPU_AVAILABLE and not FORCE_CPU

MAX_RANDOM_ENTRIES = 100
RANDOM_STATE = 765

MODEL_DEVICE = "cuda" if USE_GPU else "cpu"
MODEL_DTYPE = torch.bfloat16 if USE_GPU else torch.float32
MODEL_DEVICE_MAP = "auto" if USE_GPU else None

print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}; CUDA build: {torch.version.cuda}")
print(f"GPU available: {GPU_AVAILABLE}")
print(f"Using device: {MODEL_DEVICE}")
print(f"Model dtype: {MODEL_DTYPE}")
print(f"Device map: {MODEL_DEVICE_MAP}")
if GPU_AVAILABLE:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")


Python: c:\Users\money\anaconda3\envs\meddemo\python.exe
PyTorch: 2.11.0+cu128; CUDA build: 12.8
GPU available: True
Using device: cuda
Model dtype: torch.bfloat16
Device map: auto
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA memory: 0.0 MiB allocated, 0.0 MiB reserved


## 9. Model Evaluation Cells

The following three cells test the three locally installed models with the same basic display pattern:

- print model/device information
- show progress with `tqdm`
- report the same counter keys
- return the same result-table columns

MedGemma uses constrained-choice scoring over ESI labels. MedSigLIP is a retrieval baseline: it embeds reference/query rows, then predicts from nearest reference labels.

In [14]:
# Shared model-evaluation display settings
from IPython.display import display
from tqdm.auto import tqdm

RESULT_COLUMNS = ["model", "row", "actual", "prediction", "status", "model_output", "details"]

def show_run_header(model_id, rows_count):
    print(f"Model: {model_id}")
    print(f"Evaluation rows: {rows_count}")
    print(f"Device setting: {MODEL_DEVICE}")

def show_results(counts, rows):
    display(pd.DataFrame([dict(counts)]))
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)

eval_df = pd.read_csv(PROCESSED_DIR / "clinical_triage_cleaned_with_label.csv")
eval_rows = eval_df.sample(
    n=min(MAX_RANDOM_ENTRIES, len(eval_df)),
    random_state=RANDOM_STATE,
).reset_index(drop=True)

print(f"Shared evaluation sample: {len(eval_rows)} rows")

Shared evaluation sample: 10 rows


# Model 1: Medgemma 4B

In [29]:
# Model 1: google/medgemma-1.5-4b-it
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MEDGEMMA_4B_MODEL_ID = "google/medgemma-1.5-4b-it"

show_run_header(MEDGEMMA_4B_MODEL_ID, len(eval_rows))

processor_4b = AutoProcessor.from_pretrained(MEDGEMMA_4B_MODEL_ID, local_files_only=True)
model_4b = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_4B_MODEL_ID,
    local_files_only=True,
    torch_dtype=MODEL_DTYPE,
    device_map=MODEL_DEVICE_MAP,
)
model_4b.eval()
print("Model placement:", getattr(model_4b, "hf_device_map", "single-device"))
print("First parameter device:", next(model_4b.parameters()).device)
if torch.cuda.is_available():
    print(f"CUDA memory after load: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")

counts_4b = empty_result_counter()
rows_4b = []
for row_index, row in tqdm(eval_rows.iterrows(), total=len(eval_rows), desc="MedGemma 4B cases", unit="case"):
    actual = int(row["label_esi_level"])
    if not is_valid_model_input(row):
        counts_4b["invalid_input"] += 1
        rows_4b.append({
            "model": MEDGEMMA_4B_MODEL_ID,
            "row": row_index,
            "actual": actual,
            "prediction": None,
            "status": "invalid_input",
            "model_output": None,
            "details": {},
        })
        continue
    try:
        prompt = build_esi_prompt(row_to_case_json(row), PROMPT_TEMPLATE_PATH)
        prediction, raw_response = generate_medgemma_4b_prediction(prompt, processor_4b, model_4b)
        output = {"predicted_esi_level": prediction}
    except Exception as exc:
        prediction = "invalid_output"
        raw_response = ""
        output = f"ERROR: {exc}"
    status = score_prediction(prediction, actual, counts_4b)
    rows_4b.append({
        "model": MEDGEMMA_4B_MODEL_ID,
        "row": row_index,
        "actual": actual,
        "prediction": prediction,
        "status": status,
        "model_output": output,
        "details": {"raw_response": raw_response},
    })

show_results(counts_4b, rows_4b)

Model: google/medgemma-1.5-4b-it
Evaluation rows: 10
Device setting: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model placement: {'model.vision_tower': 0, 'model.multi_modal_projector': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 0, 'model.language_model.layers.12': 0, 'model.language_model.layers.13': 0, 'model.language_model.layers.14': 0, 'model.language_model.layers.15': 0, 'model.language_model.layers.16': 0, 'model.language_model.layers.17': 0, 'model.language_model.layers.18': 0, 'model.language_model.layers.19': 0, 'model.language_model.layers.20': 0, 'model.language_model.layers.21': 0, 'model.language_model.layers.22': 'cpu', 'model.language_model.la

MedGemma 4B cases:   0%|          | 0/10 [00:00<?, ?case/s]

,correct,incorrect,uncertain,invalid_output,invalid_input
0,7,3,0,0,0


,model,row,actual,prediction,status,model_output,details
0,google/medgemma-1.5-4b-it,0,2,2,correct,{'predicted_esi_level': 2},{'raw_response': '2'}
1,google/medgemma-1.5-4b-it,1,4,3,incorrect,{'predicted_esi_level': 3},{'raw_response': '3'}
2,google/medgemma-1.5-4b-it,2,4,3,incorrect,{'predicted_esi_level': 3},{'raw_response': '3'}
3,google/medgemma-1.5-4b-it,3,3,3,correct,{'predicted_esi_level': 3},{'raw_response': '3 ```json {'}
4,google/medgemma-1.5-4b-it,4,2,2,correct,{'predicted_esi_level': 2},{'raw_response': '2'}
5,google/medgemma-1.5-4b-it,5,4,3,incorrect,{'predicted_esi_level': 3},{'raw_response': '3'}
6,google/medgemma-1.5-4b-it,6,3,3,correct,{'predicted_esi_level': 3},{'raw_response': '3 ```json {'}
7,google/medgemma-1.5-4b-it,7,2,2,correct,{'predicted_esi_level': 2},{'raw_response': '2'}
8,google/medgemma-1.5-4b-it,8,2,2,correct,{'predicted_esi_level': 2},{'raw_response': '2'}
9,google/medgemma-1.5-4b-it,9,3,3,correct,{'predicted_esi_level': 3},{'raw_response': '3 ```json {'}


# Model 2: MedGemma 27b

In [ ]:
# Model 2: google/medgemma-27b-text-it
# This model is very large. Keep RUN_27B_TEXT = False until you intentionally want to run it.
# WARNING: May crash your Python environment
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

RUN_27B_TEXT = False
MEDGEMMA_27B_TEXT_MODEL_ID = "google/medgemma-27b-text-it"
EVAL_SAMPLE_SIZE_27B = min(MAX_RANDOM_ENTRIES, len(eval_rows))

eval_rows_27b = eval_rows.head(EVAL_SAMPLE_SIZE_27B).copy()
show_run_header(MEDGEMMA_27B_TEXT_MODEL_ID, len(eval_rows_27b))

counts_27b = empty_result_counter()
rows_27b = []

if not RUN_27B_TEXT:
    print("Status: skipped because RUN_27B_TEXT = False. Set it to True only when you intentionally want the slow/heavy run.")
else:
    tokenizer_27b = AutoTokenizer.from_pretrained(MEDGEMMA_27B_TEXT_MODEL_ID, local_files_only=True)
    model_27b = AutoModelForCausalLM.from_pretrained(
        MEDGEMMA_27B_TEXT_MODEL_ID,
        local_files_only=True,
        torch_dtype=torch.float32,
        device_map=None,
    )
    model_27b.eval()
    print("Model placement: single-device")
    print("First parameter device:", next(model_27b.parameters()).device)

    for row_index, row in tqdm(eval_rows_27b.iterrows(), total=len(eval_rows_27b), desc="MedGemma 27B cases", unit="case"):
        actual = int(row["label_esi_level"])
        if not is_valid_model_input(row):
            counts_27b["invalid_input"] += 1
            rows_27b.append({
                "model": MEDGEMMA_27B_TEXT_MODEL_ID,
                "row": row_index,
                "actual": actual,
                "prediction": None,
                "status": "invalid_input",
                "model_output": None,
                "details": {},
            })
            continue
        try:
            prompt = build_esi_prompt(row_to_case_json(row), PROMPT_TEMPLATE_PATH)
            prediction, raw_response = generate_text_model_prediction(prompt, tokenizer_27b, model_27b)
            output = {"predicted_esi_level": prediction}
        except Exception as exc:
            prediction = "invalid_output"
            raw_response = ""
            output = f"ERROR: {exc}"
        status = score_prediction(prediction, actual, counts_27b)
        rows_27b.append({
            "model": MEDGEMMA_27B_TEXT_MODEL_ID,
            "row": row_index,
            "actual": actual,
            "prediction": prediction,
            "status": status,
            "model_output": output,
            "details": {"raw_response": raw_response},
        })

show_results(counts_27b, rows_27b)

Model: google/medgemma-27b-text-it
Evaluation rows: 10
Device setting: cuda


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

: 

# Model 3: Medsiglip

In [15]:
# Model 3: google/medsiglip-448 retrieval-based ESI prediction
import numpy as np
from transformers import AutoModel, AutoProcessor

MEDSIGLIP_MODEL_ID = "google/medsiglip-448"
REFERENCE_SIZE = 200
MEDSIGLIP_EVAL_SIZE = min(MAX_RANDOM_ENTRIES, len(eval_rows))
MEDSIGLIP_BATCH_SIZE = 8
TOP_K = 5

medsiglip_df = eval_df.sample(
    n=min(REFERENCE_SIZE + MEDSIGLIP_EVAL_SIZE, len(eval_df)),
    random_state=RANDOM_STATE,
).reset_index(drop=True)
reference_df = medsiglip_df.iloc[: min(REFERENCE_SIZE, len(medsiglip_df) - 1)].copy()
query_df = medsiglip_df.iloc[len(reference_df) : len(reference_df) + MEDSIGLIP_EVAL_SIZE].copy()

show_run_header(MEDSIGLIP_MODEL_ID, len(query_df))
print(f"Reference rows: {len(reference_df)}")
print(f"Embedding batch size: {MEDSIGLIP_BATCH_SIZE}")

processor_siglip = AutoProcessor.from_pretrained(MEDSIGLIP_MODEL_ID, local_files_only=True)
model_siglip = AutoModel.from_pretrained(MEDSIGLIP_MODEL_ID, local_files_only=True)
model_siglip.to(MODEL_DEVICE)
model_siglip.eval()
print("Model placement: single-device")
print("First parameter device:", next(model_siglip.parameters()).device)
if torch.cuda.is_available():
    print(f"CUDA memory after load: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")

reference_texts = [case_json_to_retrieval_text(row_to_case_json(row)) for _, row in reference_df.iterrows()]
query_texts = [case_json_to_retrieval_text(row_to_case_json(row)) for _, row in query_df.iterrows()]
reference_embeddings = embed_texts_medsiglip(
    reference_texts,
    processor_siglip,
    model_siglip,
    batch_size=MEDSIGLIP_BATCH_SIZE,
    desc="MedSigLIP reference embeddings",
)
query_embeddings = embed_texts_medsiglip(
    query_texts,
    processor_siglip,
    model_siglip,
    batch_size=MEDSIGLIP_BATCH_SIZE,
    desc="MedSigLIP query embeddings",
)
if torch.cuda.is_available():
    print(f"CUDA memory after embeddings: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB allocated, {torch.cuda.memory_reserved(0) / 1024**2:.1f} MiB reserved")

counts_siglip = empty_result_counter()
rows_siglip = []
reference_labels = reference_df["label_esi_level"].astype(int).to_numpy()

for row_offset, (_, row) in tqdm(enumerate(query_df.iterrows()), total=len(query_df), desc="MedSigLIP cases", unit="case"):
    actual = int(row["label_esi_level"])
    if not is_valid_model_input(row):
        counts_siglip["invalid_input"] += 1
        rows_siglip.append({
            "model": MEDSIGLIP_MODEL_ID,
            "row": int(row_offset),
            "actual": actual,
            "prediction": None,
            "status": "invalid_input",
            "model_output": None,
            "details": {},
        })
        continue
    scores = reference_embeddings @ query_embeddings[row_offset]
    top_indices = np.argsort(-scores)[:TOP_K]
    top_labels = reference_labels[top_indices]
    values, vote_counts = np.unique(top_labels, return_counts=True)
    prediction = int(values[np.argmax(vote_counts)])
    output = {"predicted_esi_level": prediction}
    status = score_prediction(prediction, actual, counts_siglip)
    rows_siglip.append({
        "model": MEDSIGLIP_MODEL_ID,
        "row": int(row_offset),
        "actual": actual,
        "prediction": prediction,
        "status": status,
        "model_output": output,
        "details": {
            "top_k_labels": top_labels.tolist(),
            "top_score": float(scores[top_indices[0]]),
        },
    })

show_results(counts_siglip, rows_siglip)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Model: google/medsiglip-448
Evaluation rows: 10
Device setting: cuda
Reference rows: 200
Embedding batch size: 8


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Model placement: single-device
First parameter device: cuda:0
CUDA memory after load: 3351.6 MiB allocated, 3822.0 MiB reserved


MedSigLIP reference embeddings:   0%|          | 0/25 [00:00<?, ?it/s]

MedSigLIP query embeddings:   0%|          | 0/2 [00:00<?, ?it/s]

CUDA memory after embeddings: 3360.7 MiB allocated, 3850.0 MiB reserved


MedSigLIP cases:   0%|          | 0/10 [00:00<?, ?case/s]

,correct,incorrect,uncertain,invalid_output,invalid_input
0,10,0,0,0,0


,model,row,actual,prediction,status,model_output,details
0,google/medsiglip-448,0,3,3,correct,{'predicted_esi_level': 3},"{'top_k_labels': [3, 3, 3, 3, 3], 'top_score':..."
1,google/medsiglip-448,1,3,3,correct,{'predicted_esi_level': 3},"{'top_k_labels': [3, 3, 3, 3, 3], 'top_score':..."
2,google/medsiglip-448,2,3,3,correct,{'predicted_esi_level': 3},"{'top_k_labels': [3, 3, 3, 3, 3], 'top_score':..."
3,google/medsiglip-448,3,4,4,correct,{'predicted_esi_level': 4},"{'top_k_labels': [4, 4, 4, 4, 4], 'top_score':..."
4,google/medsiglip-448,4,3,3,correct,{'predicted_esi_level': 3},"{'top_k_labels': [3, 3, 3, 3, 3], 'top_score':..."
5,google/medsiglip-448,5,3,3,correct,{'predicted_esi_level': 3},"{'top_k_labels': [3, 3, 3, 3, 3], 'top_score':..."
6,google/medsiglip-448,6,2,2,correct,{'predicted_esi_level': 2},"{'top_k_labels': [2, 2, 2, 2, 2], 'top_score':..."
7,google/medsiglip-448,7,4,4,correct,{'predicted_esi_level': 4},"{'top_k_labels': [4, 4, 4, 4, 4], 'top_score':..."
8,google/medsiglip-448,8,2,2,correct,{'predicted_esi_level': 2},"{'top_k_labels': [2, 2, 2, 2, 2], 'top_score':..."
9,google/medsiglip-448,9,3,3,correct,{'predicted_esi_level': 3},"{'top_k_labels': [3, 3, 3, 3, 3], 'top_score':..."
